# Bronze → Silver: Clean & Transform Fact Table
Fix all data quality issues in the raw orders table.

In [0]:
from pyspark.sql.types import IntegerType, FloatType, DateType, TimestampType
import pyspark.sql.functions as F

catalog_name = 'quickbite'

In [0]:
df = spark.table(f"{catalog_name}.bronze.brz_orders")
print("Row count:", df.count())
df.printSchema()

### Step 1 — Remove duplicate orders
A duplicate is defined as the same `order_id` + `item_seq` combination.

In [0]:
dupes = df.groupBy("order_id","item_seq").count().filter(F.col("count") > 1)
print("Duplicate (order_id, item_seq) combos:", dupes.count())

df = df.dropDuplicates(["order_id", "item_seq"])
print("Rows after dedup:", df.count())

### Step 2 — Fix `item_price` (remove ₹ symbol, trim spaces, cast to float)

In [0]:
df.select("item_price").show(10, truncate=False)

In [0]:
df = df.withColumn(
    "item_price",
    F.regexp_replace(F.col("item_price"), "[₹\\s]", "").cast(FloatType())
)
df.select("item_price").show(5)

### Step 3 — Fix `delivery_time_mins` (negative values → abs)

In [0]:
neg_count = df.filter(F.col("delivery_time_mins").cast(IntegerType()) < 0).count()
print(f"Negative delivery_time_mins rows: {neg_count}")

df = df.withColumn(
    "delivery_time_mins",
    F.abs(F.col("delivery_time_mins").cast(IntegerType()))
)

### Step 4 — Fill NULL `agent_id` with 'UNASSIGNED'

In [0]:
null_agents = df.filter(F.col("agent_id").isNull()).count()
print(f"NULL agent_id rows: {null_agents}")

df = df.fillna("UNASSIGNED", subset=["agent_id"])

### Step 5 — Standardise `city` casing

In [0]:
df = df.withColumn("city", F.initcap(F.col("city")))
df.select("city").distinct().orderBy("city").show()

### Step 6 — Type conversions

In [0]:
df = df.withColumn("dt",                 F.to_date(F.col("dt"), "yyyy-MM-dd")) \
           .withColumn("order_ts",           F.to_timestamp(F.col("order_ts"), "yyyy-MM-dd HH:mm:ss")) \
           .withColumn("item_seq",           F.col("item_seq").cast(IntegerType())) \
           .withColumn("quantity",           F.col("quantity").cast(IntegerType())) \
           .withColumn("discount_pct",       F.col("discount_pct").cast(FloatType())) \
           .withColumn("delivery_fee",       F.col("delivery_fee").cast(FloatType()))

df.printSchema()

### Step 7 — Final check

In [0]:
display(df.limit(10))

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_orders")

print("slv_orders written:", spark.table(f"{catalog_name}.silver.slv_orders").count(), "rows")